In [1]:
import os
import anndata as ad
import pandas as pd
sc_dir = '/work/magroup/skrieger/Syndri/data/datasets/brain_merfish/raw/single-cell'
st_data = ad.read_h5ad('/work/magroup/skrieger/Syndri/data/datasets/brain_merfish/interim/spatial/Zhuang-ABCA-4_aggregated.h5ad')
metadata = pd.read_csv(os.path.join(sc_dir,'cell_metadata_with_cluster_annotation.csv'))
gene_set = st_data.var['gene_identifier']
# del st_data
v3_files = sorted([f for f in os.listdir(sc_dir) if '10Xv3' in f])
v3_files

/tmp/ipykernel_1098595/419267943.py:6: DtypeWarning: Columns (0: neurotransmitter) have mixed types. Specify dtype option on import or set low_memory=False.
  metadata = pd.read_csv(os.path.join(sc_dir,'cell_metadata_with_cluster_annotation.csv'))


['WMB-10Xv3-CB-raw.h5ad',
 'WMB-10Xv3-CTXsp-raw.h5ad',
 'WMB-10Xv3-HPF-raw.h5ad',
 'WMB-10Xv3-HY-raw.h5ad',
 'WMB-10Xv3-Isocortex-1-raw.h5ad',
 'WMB-10Xv3-Isocortex-2-raw.h5ad',
 'WMB-10Xv3-MB-raw.h5ad',
 'WMB-10Xv3-MY-raw.h5ad',
 'WMB-10Xv3-OLF-raw.h5ad',
 'WMB-10Xv3-P-raw.h5ad',
 'WMB-10Xv3-PAL-raw.h5ad',
 'WMB-10Xv3-STR-raw.h5ad',
 'WMB-10Xv3-TH-raw.h5ad']

In [2]:
import numpy as np
spatial_counts = (
    st_data.obs
    .groupby(["class", "cluster"], observed=True)
    .size()
    .reset_index(name="cell_count")
)
n_total = 300_000

class_counts = (
    spatial_counts
    .groupby("class", observed=True)["cell_count"]
    .sum()
)

class_weights = np.sqrt(class_counts / class_counts.sum())
class_weights = class_counts / class_counts.sum()
class_weights /= class_weights.sum()

class_targets = (class_weights * n_total).round().astype(int)

cluster_targets = {}

for class_name, df in spatial_counts.groupby("class", observed=True):

    weights = np.sqrt(
        df["cell_count"] / df["cell_count"].sum()
    )
    weights /= weights.sum()

    n_class = class_targets[class_name]

    targets = np.round(weights * n_class).astype(int)

    for cluster, n in zip(df["cluster"], targets):
        cluster_targets[cluster] = n

cluster_targets



{'0001 CLA-EPd-CTX Car3 Glut_1': 380,
 '0002 CLA-EPd-CTX Car3 Glut_1': 337,
 '0003 CLA-EPd-CTX Car3 Glut_1': 525,
 '0004 CLA-EPd-CTX Car3 Glut_1': 727,
 '0005 CLA-EPd-CTX Car3 Glut_2': 207,
 '0006 IT EP-CLA Glut_1': 116,
 '0007 IT EP-CLA Glut_1': 611,
 '0008 IT EP-CLA Glut_1': 64,
 '0009 IT EP-CLA Glut_1': 150,
 '0010 IT EP-CLA Glut_1': 52,
 '0011 IT EP-CLA Glut_1': 101,
 '0012 IT EP-CLA Glut_1': 82,
 '0013 IT EP-CLA Glut_2': 26,
 '0014 IT EP-CLA Glut_2': 133,
 '0017 IT EP-CLA Glut_2': 202,
 '0018 IT EP-CLA Glut_3': 145,
 '0019 IT EP-CLA Glut_3': 152,
 '0020 IT EP-CLA Glut_3': 26,
 '0022 IT EP-CLA Glut_4': 26,
 '0024 L5/6 IT TPE-ENT Glut_1': 107,
 '0026 L5/6 IT TPE-ENT Glut_2': 94,
 '0029 L5/6 IT TPE-ENT Glut_3': 45,
 '0030 L5/6 IT TPE-ENT Glut_4': 277,
 '0033 L6 IT CTX Glut_1': 26,
 '0034 L6 IT CTX Glut_1': 116,
 '0036 L6 IT CTX Glut_1': 152,
 '0037 L6 IT CTX Glut_1': 296,
 '0038 L6 IT CTX Glut_2': 64,
 '0039 L6 IT CTX Glut_2': 143,
 '0040 L6 IT CTX Glut_2': 150,
 '0041 L6 IT CTX Glut

In [4]:
drop_cols = ['cell_barcode','library_label','anatomical_division_label']

In [5]:
import anndata as ad
import numpy as np
import pandas as pd


celltype_col = "cluster"
obs_tables = []

metadata_indexed = metadata.set_index("cell_label")

# ---------------------------------------------------------
# 1. Read metadata only
# ---------------------------------------------------------
for file_idx, path in enumerate(v3_files):

    adata = ad.read_h5ad(
        os.path.join(sc_dir, path),
        backed="r",
    )

    # Keep original AnnData index intact
    obs = adata.obs.copy()

    obs = obs.drop(columns=drop_cols).join(
        metadata_indexed,
        how="left",
    )

    obs = obs[[celltype_col, "class"]].copy()
    obs["obs_name"] = adata.obs_names.to_numpy()
    obs["file_idx"] = file_idx

    obs_tables.append(obs)

    adata.file.close()

all_obs = pd.concat(
    obs_tables,
    ignore_index=True,
)

available_counts = all_obs["cluster"].value_counts()

cluster_targets = {
    cluster: min(
        n,
        available_counts.get(cluster, 0)
    )
    for cluster, n in cluster_targets.items()
}

missing_clusters = [
    cluster
    for cluster in spatial_counts["cluster"].unique()
    if cluster not in available_counts.index
]

print("Spatial clusters absent from SC reference:")
print(missing_clusters)


rng = np.random.default_rng(42)

sampled = []

for cluster, n in cluster_targets.items():

    candidates = all_obs[
        all_obs["cluster"] == cluster
    ]

    if len(candidates) == 0:
        continue

    idx = rng.choice(
        len(candidates),
        size=min(n, len(candidates)),
        replace=False,
    )

    sampled.append(candidates.iloc[idx])

sampled_obs = pd.concat(
    sampled,
    ignore_index=True,
)

Spatial clusters absent from SC reference:
['4808 PSV Pax2 Gly-Gaba_1', '5195 CBX Purkinje Gaba_2']


In [6]:
sampled_obs['file_idx'].value_counts()

file_idx
0     66350
2     39196
7     29598
4     28903
5     22994
11    21080
1     14408
6     12978
8      9973
9      9709
10     6090
12     4951
3      4934
Name: count, dtype: int64

In [7]:
st_data[st_data.obs['cluster'].isin(missing_clusters)]

View of AnnData object with n_obs × n_vars = 8 × 1122
    obs: 'brain_section_label', 'feature_matrix_label', 'donor_label', 'donor_genotype', 'donor_sex', 'cluster_alias', 'x', 'y', 'z', 'subclass_confidence_score', 'cluster_confidence_score', 'high_quality_transfer', 'neurotransmitter', 'class', 'subclass', 'supertype', 'cluster', 'neurotransmitter_color', 'class_color', 'subclass_color', 'supertype_color', 'cluster_color'
    var: 'gene_symbol', 'gene_identifier', 'name', 'mapped_ncbi_identifier'
    layers: None (.X)

In [8]:
# ---------------------------------------------------------
# 2. Reopen each file and load ONLY the sampled cells
# ---------------------------------------------------------
sampled_adatas = []

for file_idx, path in enumerate(sorted(v3_files)):
    print(path)
    # Cells selected from this particular file
    selected = sampled_obs.loc[
        sampled_obs["file_idx"] == file_idx,
        "obs_name",
    ]

    if len(selected) == 0:
        continue

    adata = ad.read_h5ad(
        os.path.join(sc_dir, path),
        backed="r",
    )

    # Convert obs names -> integer row positions
    idx = adata.obs_names.get_indexer(selected)

    if np.any(idx == -1):
        missing = selected[idx == -1]
        raise ValueError(
            f"{len(missing)} selected cells were not found in {path}"
        )

    # Only these cells are brought into memory
    subset = adata[idx].to_memory()
    adata.file.close()
    
    # Add metadata to the resulting subset
    subset.obs = subset.obs.drop(columns=drop_cols).join(
        metadata.set_index("cell_label"),
        how="left",
    )

    sampled_adatas.append(subset)

    

WMB-10Xv3-CB-raw.h5ad
WMB-10Xv3-CTXsp-raw.h5ad
WMB-10Xv3-HPF-raw.h5ad
WMB-10Xv3-HY-raw.h5ad
WMB-10Xv3-Isocortex-1-raw.h5ad
WMB-10Xv3-Isocortex-2-raw.h5ad
WMB-10Xv3-MB-raw.h5ad
WMB-10Xv3-MY-raw.h5ad
WMB-10Xv3-OLF-raw.h5ad
WMB-10Xv3-P-raw.h5ad
WMB-10Xv3-PAL-raw.h5ad
WMB-10Xv3-STR-raw.h5ad
WMB-10Xv3-TH-raw.h5ad


In [9]:
# ---------------------------------------------------------
# 3. Combine the sampled references
# ---------------------------------------------------------
sc_reference = ad.concat(
    sampled_adatas,
    axis=0,
    join="inner",
    merge="same",
    index_unique=None,
)
del sampled_adatas

print(sc_reference)

AnnData object with n_obs × n_vars = 271164 × 32285
    obs: 'cell_barcode', 'barcoded_cell_sample_label', 'library_label', 'feature_matrix_label', 'entity', 'brain_section_label', 'library_method', 'region_of_interest_acronym', 'donor_label', 'donor_genotype', 'donor_sex', 'dataset_label', 'x', 'y', 'cluster_alias', 'neurotransmitter', 'class', 'subclass', 'supertype', 'cluster', 'neurotransmitter_color', 'class_color', 'subclass_color', 'supertype_color', 'cluster_color', 'region_of_interest_order', 'region_of_interest_color'
    var: 'gene_symbol'
    layers: None (.X)


In [18]:
gene_set = list(st_data.var['gene_identifier'].values)
gene_set

['ENSMUSG00000024798',
 'ENSMUSG00000042385',
 'ENSMUSG00000036198',
 'ENSMUSG00000028780',
 'ENSMUSG00000015843',
 'ENSMUSG00000026768',
 'ENSMUSG00000049928',
 'ENSMUSG00000041046',
 'ENSMUSG00000032373',
 'ENSMUSG00000004633',
 'ENSMUSG00000022468',
 'ENSMUSG00000026514',
 'ENSMUSG00000037681',
 'ENSMUSG00000051910',
 'ENSMUSG00000005672',
 'ENSMUSG00000032649',
 'ENSMUSG00000050315',
 'ENSMUSG00000027568',
 'ENSMUSG00000046719',
 'ENSMUSG00000031130',
 'ENSMUSG00000020182',
 'ENSMUSG00000046546',
 'ENSMUSG00000050783',
 'ENSMUSG00000000792',
 'ENSMUSG00000079330',
 'ENSMUSG00000035910',
 'ENSMUSG00000023232',
 'ENSMUSG00000026639',
 'ENSMUSG00000027859',
 'ENSMUSG00000005892',
 'ENSMUSG00000021919',
 'ENSMUSG00000044317',
 'ENSMUSG00000021032',
 'ENSMUSG00000049436',
 'ENSMUSG00000040016',
 'ENSMUSG00000029123',
 'ENSMUSG00000036357',
 'ENSMUSG00000026668',
 'ENSMUSG00000071984',
 'ENSMUSG00000053747',
 'ENSMUSG00000033740',
 'ENSMUSG00000050377',
 'ENSMUSG00000091002',
 'ENSMUSG00

In [19]:
sc_reference = sc_reference[:,gene_set]

In [21]:
sc_reference.write_h5ad('/work/magroup/skrieger/Syndri/data/datasets/brain_merfish/interim/single-cell/10Xv3_subsampled.h5ad')